In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/NIH_Chest_XRAY/Data_Entry_2017.csv')

#df_pos = df[df['Finding Labels'].str.contains('Pneumonia')]
#df_neg = df[df['Finding Labels']=='No Finding']
#df_neg = df_neg[df_neg['Follow-up #']==0]

In [9]:
df_pos = df[df['Finding Labels'].str.contains('Pneumonia')]
# keep the first image with Pneumonia per Patient ID
df_pos = df_pos.loc[df_pos.groupby('Patient ID')['Follow-up #'].idxmin()]
display(df_pos.describe())
df_pos

,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
count,1008.000000,1008.000000,1008.000000,1008.000000,1008.000000,1008.000000,1008.000000,0.0
mean,7.768849,14566.480159,45.900794,2675.726190,2481.670635,0.154204,0.154204,NaN
std,10.934723,8336.535550,21.084784,337.858344,394.145277,0.015311,0.015311,NaN
min,0.000000,13.000000,2.000000,1581.000000,1537.000000,0.139000,0.139000,NaN
25%,1.000000,7774.000000,32.000000,2500.000000,2048.000000,0.143000,0.143000,NaN
50%,4.000000,14432.500000,48.000000,2544.000000,2544.000000,0.143000,0.143000,NaN
75%,10.000000,20954.250000,59.000000,2992.000000,2991.000000,0.168000,0.168000,NaN
max,105.000000,30801.000000,412.000000,3056.000000,3056.000000,0.194311,0.194311,NaN


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
48,00000013_010.png,Effusion|Pneumonia|Pneumothorax,10,13,60,M,AP,3056,2544,0.139,0.139,NaN
126,00000032_012.png,Atelectasis|Consolidation|Edema|Pneumonia,12,32,55,F,AP,2500,2048,0.168,0.168,NaN
253,00000056_000.png,Nodule|Pneumonia,0,56,76,M,PA,2500,2048,0.168,0.168,NaN
276,00000061_012.png,Edema|Effusion|Infiltration|Pleural_Thickening...,12,61,77,M,AP,3056,2544,0.139,0.139,NaN
590,00000144_001.png,Pneumonia,1,144,83,M,AP,2500,2048,0.168,0.168,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
111557,00030536_007.png,Atelectasis|Consolidation|Pneumonia,7,30536,56,F,AP,3056,2544,0.139,0.139,NaN
111627,00030570_001.png,Edema|Infiltration|Pneumonia,1,30570,29,F,AP,3056,2544,0.139,0.139,NaN
111767,00030621_002.png,Pneumonia,2,30621,22,F,AP,3056,2544,0.139,0.139,NaN
111845,00030637_016.png,Consolidation|Pneumonia,16,30637,48,M,AP,3056,2544,0.139,0.139,NaN


In [10]:
# Count the number of follow-ups per patient
followup_counts = df.groupby('Patient ID')['Follow-up #'].count().reset_index()
followup_counts.rename(columns={'Follow-up #': 'FollowupCount'}, inplace=True)

# Filter for patients with only one follow-up
single_followup_patients = followup_counts[followup_counts['FollowupCount'] == 1]['Patient ID']

# Filter the original DataFrame to keep only these patients and where Finding Labels is 'No Finding'
df_neg = df[(df['Patient ID'].isin(single_followup_patients)) & (df['Finding Labels'] == 'No Finding')]

display(df_neg.describe())

,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
count,12788.0,12788.000000,12788.000000,12788.000000,12788.000000,12788.000000,12788.000000,0.0
mean,0.0,16162.347748,43.348608,2569.572490,2576.200109,0.156103,0.156103,NaN
std,0.0,8936.466555,16.273667,372.879907,400.605188,0.017392,0.017392,NaN
min,0.0,2.000000,1.000000,1189.000000,1167.000000,0.115000,0.115000,NaN
25%,0.0,8389.750000,31.000000,2292.000000,2048.000000,0.143000,0.143000,NaN
50%,0.0,16377.500000,44.000000,2536.000000,2544.000000,0.143000,0.143000,NaN
75%,0.0,24127.750000,55.000000,2992.000000,2991.000000,0.168000,0.168000,NaN
max,0.0,30805.000000,155.000000,3056.000000,3056.000000,0.198800,0.198800,NaN


In [5]:
df_neg = df_neg.sample(frac=0.1,random_state=42)

In [11]:
df = pd.concat([df_pos, df_neg])
df['Label'] = df['Finding Labels'].apply(lambda x: 0 if x == 'No Finding' else 1)

df = df.dropna(subset=['Patient Gender'])
df['Patient Gender'] = df['Patient Gender'].map({'M': 0, 'F': 1})

display(df.describe())
df


,Follow-up #,Patient ID,Patient Age,Patient Gender,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,Label
count,13796.000000,13796.000000,13796.000000,13796.000000,13796.000000,13796.000000,13796.000000,13796.000000,0.0,13796.000000
mean,0.567628,16045.746231,43.535083,0.470571,2577.328573,2569.293346,0.155965,0.155965,NaN,0.073065
std,3.579956,8903.409368,16.684558,0.499151,371.450656,400.878219,0.017255,0.017255,NaN,0.260252
min,0.000000,2.000000,1.000000,0.000000,1189.000000,1167.000000,0.115000,0.115000,NaN,0.000000
25%,0.000000,8337.500000,31.000000,0.000000,2370.000000,2048.000000,0.143000,0.143000,NaN,0.000000
50%,0.000000,16205.500000,44.000000,0.000000,2538.000000,2544.000000,0.143000,0.143000,NaN,0.000000
75%,0.000000,24022.250000,55.000000,1.000000,2992.000000,2991.000000,0.168000,0.168000,NaN,0.000000
max,105.000000,30805.000000,412.000000,1.000000,3056.000000,3056.000000,0.198800,0.198800,NaN,1.000000


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,Label
48,00000013_010.png,Effusion|Pneumonia|Pneumothorax,10,13,60,0,AP,3056,2544,0.139,0.139,NaN,1
126,00000032_012.png,Atelectasis|Consolidation|Edema|Pneumonia,12,32,55,1,AP,2500,2048,0.168,0.168,NaN,1
253,00000056_000.png,Nodule|Pneumonia,0,56,76,0,PA,2500,2048,0.168,0.168,NaN,1
276,00000061_012.png,Edema|Effusion|Infiltration|Pleural_Thickening...,12,61,77,0,AP,3056,2544,0.139,0.139,NaN,1
590,00000144_001.png,Pneumonia,1,144,83,0,AP,2500,2048,0.168,0.168,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
112113,00030800_000.png,No Finding,0,30800,34,1,PA,2048,2500,0.168,0.168,NaN,0
112116,00030802_000.png,No Finding,0,30802,29,0,PA,2048,2500,0.168,0.168,NaN,0
112117,00030803_000.png,No Finding,0,30803,42,1,PA,2048,2500,0.168,0.168,NaN,0
112118,00030804_000.png,No Finding,0,30804,30,1,PA,2048,2500,0.168,0.168,NaN,0


In [12]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.drop(columns=['Follow-up #', "Finding Labels", "Patient ID", "Unnamed: 11"], inplace=True)
df.rename(columns={'Image Index': 'image', "Label": "class",
                   'Patient Age': 'age', 'Patient Gender': 'gender', 'View Position': 'view_pos',
                   'OriginalImage[Width': 'img_width', 'Height]': 'img_height',
                   'OriginalImagePixelSpacing[x': 'spacing_x', 'y]': 'spacing_y'}, inplace=True)
df

,image,age,gender,view_pos,img_width,img_height,spacing_x,spacing_y,class
0,00004143_000.png,20,0,PA,2500,2048,0.171000,0.171000,0
1,00015705_000.png,25,1,PA,2048,2500,0.168000,0.168000,0
2,00020482_025.png,29,1,AP,2500,2048,0.168000,0.168000,1
3,00014041_000.png,45,1,PA,2992,2991,0.143000,0.143000,0
4,00013513_000.png,67,0,PA,2834,2677,0.143000,0.143000,0
...,...,...,...,...,...,...,...,...,...
13791,00010906_000.png,59,0,PA,2992,2991,0.143000,0.143000,0
13792,00030104_000.png,29,0,PA,2021,2021,0.194311,0.194311,0
13793,00011339_000.png,69,0,PA,2992,2991,0.143000,0.143000,0
13794,00025473_005.png,24,1,AP,3056,2544,0.139000,0.139000,1


In [ ]:
df.to_csv('../../data/NIH_Chest_XRAY/experiment_data.csv', index=False)